In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
def remap_mask(mask):

    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

def remap_mask_binary(mask):

    mask_np = mask.numpy().squeeze()

    binary_mask = (mask_np != 0).astype(np.uint8)
    return torch.from_numpy(binary_mask).unsqueeze(0)

In [ ]:
import os
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset
import numpy as np

class FloodSegmentationDataset(Dataset):

    def __init__(self, root_dir, metadata_df, transform=None, target_transform=None):
        self.root_dir = root_dir
        self.metadata = metadata_df
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):

        img_name = self.metadata.iloc[idx, 0] # 'image_filename' column
        mask_name = self.metadata.iloc[idx, 1] # 'mask_filename' column

        img_path = os.path.join(self.root_dir, "images", img_name)
        mask_path = os.path.join(self.root_dir, "masks", mask_name)

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")  # Convert mask to grayscale (1 channel)

        if self.transform:
            image = self.transform(image)

        if self.target_transform:
            mask = self.target_transform(mask)

        # Replace mask values with remapped values
        mask = remap_mask_binary(mask)

        return image, mask

In [ ]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

# Define transforms for images and masks
image_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((256, 256)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

mask_transforms = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.PILToTensor(),

])

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

## **🔹 Splitting the Dataset into Train & Test**

# Define dataset paths
dataset_root = os.path.join(path, 'dataset')
image_dir = os.path.join(dataset_root, 'images')
mask_dir = os.path.join(dataset_root, 'masks')

# List all image files and sort them to ensure corresponding masks are matched
image_files = sorted([f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png', '.jpeg'))])
mask_files = sorted([f for f in os.listdir(mask_dir) if f.endswith(('.png', '.jpeg'))])

# Ensure that the number of images and masks match
if len(image_files) != len(mask_files):
    raise ValueError("Number of image files and mask files do not match.")

# Create a DataFrame to hold image and mask paths

metadata_df = pd.DataFrame({
    'image_filename': image_files,
    'mask_filename': mask_files
})

# Split metadata DataFrame into 80% train, 20% test
train_data, test_data = train_test_split(metadata_df, test_size=0.2, random_state=42, shuffle=True)

print(f"Total samples: {len(metadata_df)}")
print(f"Training samples: {len(train_data)}")
print(f"Testing samples: {len(test_data)}")

In [ ]:
import os
import pandas as pd
from torch.utils.data import DataLoader



train_dataset = FloodSegmentationDataset(root_dir=dataset_root, metadata_df=train_data,
                                         transform=image_transforms, target_transform=mask_transforms)
test_dataset = FloodSegmentationDataset(root_dir=dataset_root, metadata_df=test_data,
                                        transform=image_transforms, target_transform=mask_transforms)


# Create Train & Test DataLoaders
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)

# Check dataset sizes
print(f"Training Samples: {len(train_dataset)}, Testing Samples: {len(test_dataset)}")

In [ ]:
import matplotlib.pyplot as plt


def denormalize(img):
    mean = np.array([0.485, 0.456, 0.406])  # ImageNet mean
    std = np.array([0.229, 0.224, 0.225])  # ImageNet std
    img = img.numpy().transpose(1, 2, 0)  # Convert to HWC
    img = img * std + mean  # Reverse normalization
    img = np.clip(img, 0, 1)  # Clip values to [0,1]
    return img

# Display some images with their masks
for i in range(4):
    img, mask = train_dataset[i]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(denormalize(img))
    axes[0].set_title("Image")
    axes[0].axis("off")
    axes[1].imshow(mask.permute(1,2,0), cmap="gray")
    axes[1].set_title("Segmentation Mask")
    axes[1].axis("off")
    plt.show()

In [ ]:
!pip install segmentation_models_pytorch

import segmentation_models_pytorch as smp

# Define U-Net Model
device = "cpu"
model = smp.Unet(
    encoder_name="efficientnet-b0",  # Pretrained encoder (backbone)
    encoder_weights="imagenet",  # Use ImageNet weights
    in_channels=3,  # RGB images
    classes=8,  # Multi-class segmentation (8 output channels for 8 classes)
).to(device)

In [ ]:
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, masks in tqdm(dataloader):
        # Masks need to be Long type and squeezed to (B, H, W) for CrossEntropyLoss
        images, masks = images.to(device), masks.to(device).long().squeeze(1)

        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, masks in dataloader:
            # Masks need to be Long type and squeezed to (B, H, W) for CrossEntropyLoss
            images, masks = images.to(device), masks.to(device).long().squeeze(1)

            outputs = model(images)
            loss = criterion(outputs, masks)
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
import torch
from torch import nn
# Define loss function and optimizer
# For multi-class segmentation, CrossEntropyLoss is typically used
criterion = nn.CrossEntropyLoss() # Changed from BCEWithLogitsLoss to CrossEntropyLoss
optimizer = optim.AdamW(model.parameters(), lr=0.0001)

num_epochs = 10  # Define number of epochs
train_losses = []
val_losses = []

# Training Loop
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Plotting the loss curve
plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs + 1), train_losses, label='Training Loss')
plt.plot(range(1, num_epochs + 1), val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss Curve')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np

# Function to denormalize images
def denormalize(img):
    mean = np.array([0.485, 0.456, 0.406])  # ImageNet mean
    std = np.array([0.229, 0.224, 0.225])  # ImageNet std
    img = img.numpy().transpose(1, 2, 0)  # Convert to HWC
    img = img * std + mean  # Reverse normalization
    img = np.clip(img, 0, 1)  # Clip values to [0,1]
    return img

# Set model to evaluation mode
model.eval()

# Get some test samples
test_samples = random.sample(range(len(test_dataset)), 5)

for idx in test_samples:
    img, mask = test_dataset[idx]

    with torch.no_grad():
        pred_logits = model(img.unsqueeze(0).to(device))  # Forward pass


    pred_mask = torch.argmax(pred_logits, dim=1).squeeze(0).cpu().numpy()

    # Display images
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Original Image (Denormalized)
    axes[0].imshow(denormalize(img))
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    # Ground Truth Mask

    axes[1].imshow(mask.squeeze(), cmap="viridis")
    axes[1].set_title("Ground Truth Mask")
    axes[1].axis("off")

    # Predicted Mask
    axes[2].imshow(pred_mask, cmap="viridis")
    axes[2].set_title("Predicted Mask")
    axes[2].axis("off")

    plt.show()